In [1]:
%pwd

'd:\\Tipto\\OmniChef-Nexus\\notebooks'

In [2]:
import os
os.chdir('../')

In [3]:
%pwd

'd:\\Tipto\\OmniChef-Nexus'

In [4]:
# load the 15k recipe file
import pandas as pd 

df = pd.read_csv("data/all csv files/recipes_15k_samples.csv")
df.shape

(15698, 15)

In [5]:
df.head(1)

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,recipe_id,rating,review,num of ratings,markdown_recipe,image_path
0,chicken lickin good pork chops,500,"['weeknight', 'time-to-make', 'course', 'main-...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"['dredge pork chops in mixture of flour , salt...",here's and old standby i enjoy from time to ti...,"['lean pork chops', 'flour', 'salt', 'dry must...",7,63986,4.368421,"[""I made this for dinner tonight and the chops...",19.0,# Chicken Lickin Good Pork Chops\n**Recipe I...,data/output/images/recipe_id_63986.png


In [7]:
for col in df.columns.to_list():
    print(f"{col} => {type(col)}")

name => <class 'str'>
minutes => <class 'str'>
tags => <class 'str'>
nutrition => <class 'str'>
n_steps => <class 'str'>
steps => <class 'str'>
description => <class 'str'>
ingredients => <class 'str'>
n_ingredients => <class 'str'>
recipe_id => <class 'str'>
rating => <class 'str'>
review => <class 'str'>
num of ratings => <class 'str'>
markdown_recipe => <class 'str'>
image_path => <class 'str'>


In [8]:
import ast 
import json

list_columns = ['tags', 'nutrition', 'steps', 'ingredients' , 'review']
for col in list_columns:
    df[col] = df[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x , str) else x
    )

In [9]:
# renaming image_path to file name so that hf ensure image preview
df = df.rename(columns = {'image_path' : 'file_name'})

In [10]:
df.head(2)

,name,minutes,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,recipe_id,rating,review,num of ratings,markdown_recipe,file_name
0,chicken lickin good pork chops,500,"[weeknight, time-to-make, course, main-ingredi...","[105.7, 8.0, 0.0, 26.0, 5.0, 4.0, 3.0]",5,"[dredge pork chops in mixture of flour , salt ...",here's and old standby i enjoy from time to ti...,"[lean pork chops, flour, salt, dry mustard, ga...",7,63986,4.368421,[I made this for dinner tonight and the chops ...,19.0,# Chicken Lickin Good Pork Chops\n**Recipe I...,data/output/images/recipe_id_63986.png
1,chile rellenos,45,"[60-minutes-or-less, time-to-make, course, mai...","[94.0, 10.0, 0.0, 11.0, 11.0, 21.0, 0.0]",9,"[drain green chiles, sprinkle cornstarch on sh...",a favorite from a local restaurant no longer i...,"[egg roll wrap, whole green chilies, cheese, c...",5,43026,4.045455,"[Grandma Pam, Oh my goodness,these were so eas...",22.0,# Chile Rellenos\n**Recipe ID:** 43026\n**Cook...,data/output/images/recipe_id_43026.png


### Prepare dataset for upload to Hugging Face

For uploading dataset to we need:
1. Folder of Images
2. `jsonl` file in the folder with `file_name` field point

In [4]:
OUTPUT_DIR  = "data/output/images"   
OUTPUT_JSONL = os.path.join(OUTPUT_DIR, "metadata.jsonl")

In [11]:
from pathlib import Path

count = 0
for _ , row in df.iterrows():
    count += 1
    
    file_path = row.get("file_name")
    file_name = Path(str(file_path)).name 
    full_image_path = Path(OUTPUT_DIR)/file_name
    full_image_path = full_image_path.as_posix()
    print(full_image_path)
    if count == 5:
        break

data/output/images/recipe_id_63986.png
data/output/images/recipe_id_43026.png
data/output/images/recipe_id_23933.png
data/output/images/recipe_id_54100.png
data/output/images/recipe_id_67664.png


In [12]:
written = 0
skipped = 0

with open(OUTPUT_JSONL , "w" , encoding = 'utf-8') as f:
    for _ , row in df.iterrows():
        file_path = row.get("file_name" , "")
        file_name = Path(str(file_path)).name 
        full_image_path = Path(OUTPUT_DIR)/file_name
        full_image_path = full_image_path.as_posix()
        
        # skip rows whose file doesn't exists
        if not os.path.exists(full_image_path):
            skipped += 1
            continue
        
        record = {
            "file_name" : file_name,
            "name" : row.get("name"),
            "minutes" : int(row["minutes"]) if pd.notna(row.get("minutes")) else None,
            "tags" : row.get("tags"),
            "nutrition" : row.get("nutrition"),
            "n_steps" : int(row["n_steps"]) if pd.notna(row.get("n_steps")) else None,
            "steps" : row.get("steps"),
            "description" : row.get("description"),
            "ingredients" : row.get("ingredients"),
            "n_ingredients" : int(row["n_ingredients"]) if pd.notna(row.get("n_ingredients")) else None,
            "recipe_id" : int(row["recipe_id"]) if pd.notna(row.get("recipe_id")) else None,
            "rating" : float(row["rating"]) if pd.notna(row.get("rating")) else None,
            "review" : row.get("review"),
            "num_of_ratings" : float(row["num_of_ratings"]) if pd.notna(row.get("num_of_ratings")) else None,
            "markdown_recipe": row.get("markdown_recipe"),
        }
        
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        written += 1
        

print(f"written : {written:,} rows")
print(f"skipped : {skipped:,} rows (image file missing)")
print(f"Output  : {OUTPUT_JSONL}")     

written : 15,698 rows
skipped : 0 rows (image file missing)
Output  : data/output/images\metadata.jsonl


* created the repo on hugging face.

In [4]:
import shutil
import json
from pathlib import Path

In [5]:
IMAGE_DIR = "data/output/images"
METADATA_FILE = "data/output/images/metadata.jsonl"
UPLOAD_TEMP_DIR = "data/output/hf_upload_temp" # a copy for HF repo
os.path.exists(IMAGE_DIR) , os.path.exists(METADATA_FILE)

(True, True)

In [8]:
SHARD_SIZE = 2000 
IMAGE_SUBFOLDER = "data"

In [17]:
def prepare_sharded_copy():
    src_root = Path(IMAGE_DIR)
    dst_root = Path(UPLOAD_TEMP_DIR)
    
    # Create the temporary upload folder
    dst_root.mkdir(parents=True, exist_ok=True)
    
    # Read original metadata
    new_metadata = []
    with open(METADATA_FILE, 'r') as f:
        rows = [json.loads(line) for line in f]

    print(f"Copying and sharding {len(rows)} files to {UPLOAD_TEMP_DIR}...")

    for i, row in enumerate(rows):
        # Extract filename (handle case where metadata might already have paths)
        old_filename = os.path.basename(row['file_name'])
        
        shard_idx = i // SHARD_SIZE
        shard_folder = dst_root / IMAGE_SUBFOLDER / f"shard_{shard_idx}"
        shard_folder.mkdir(parents=True, exist_ok=True)
        
        src_path = src_root / old_filename
        new_rel_path = f"{IMAGE_SUBFOLDER}/shard_{shard_idx}/{old_filename}"
        dst_path = dst_root / new_rel_path
        
        if src_path.exists():
            shutil.copy2(src_path, dst_path) # Preserves original file metadata
            row['file_name'] = new_rel_path
            new_metadata.append(row)
        else:
            print(f"Skipping: {old_filename} (not found in {IMAGE_DIR})")

    # Write the new metadata.jsonl to the root of the upload folder
    with open(dst_root / "metadata.jsonl", 'w') as f:
        for row in new_metadata:
            f.write(json.dumps(row) + "\n")

    print("\nSuccess!")
    print(f"Original folder untouched: {IMAGE_DIR}")
    print(f"Sharded copy ready: {UPLOAD_TEMP_DIR}")

In [18]:
prepare_sharded_copy()

Copying and sharding 15698 files to data/output/hf_upload_temp...

Success!
Original folder untouched: data/output/images
Sharded copy ready: data/output/hf_upload_temp


In [6]:
from huggingface_hub import HfApi
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
REPO_ID = "tiptoghosh/food-recipes-15k" # hugging face repo id
HF_TOKEN = os.getenv("HF_TOKEN") # hugging face token
UPLOAD_TEMP_DIR = "data/output/hf_upload_temp" # folder to push on HF

In [8]:
api = HfApi(token = HF_TOKEN)

In [9]:
api.upload_large_folder(
    repo_id = REPO_ID,
    folder_path = UPLOAD_TEMP_DIR,
    repo_type = "dataset"
)

Recovering from metadata files:   0%|          | 0/15699 [00:00<?, ?it/s]




---------- 2026-04-17 10:01:33 (0:00:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 1536/15699 (927.2M/7.7G) | committed: 225/15699 (107.5M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 7 | committing: 1 | waiting: 0
---------------------------------------------------


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


---------- 2026-04-17 10:03:33 (0:02:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 5120/15699 (2.7G/7.7G) | committed: 550/15699 (452.9M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 7 | committing: 1 | waiting: 0
---------------------------------------------------
                       
---------- 2026-04-17 10:04:33 (0:03:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 5120/15699 (2.7G/7.7G) | committed: 550/15699 (452.9M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 7 | committing: 1 | waiting: 0
---------------------------------------------------
                       
---------- 2026-04-17 10:05:33 (0:04:00) ----------
Files:   hashed 15699/15699 (7.7G/7.7G) | pre-uploaded: 5120/15699 (2.7G/7.7G) | committed: 550/15699 (452.9M/7.7G) | ignored: 0
Workers: hashing: 0 | get upload mode: 0 | pre-uploading: 7 | committing: 1 | waiting: 0
-------------------------------------

KeyboardInterrupt: 

In [ ]:
api.upload_file(
    path_or_fileobj= "data/output/hf_upload_temp/metadata.jsonl",
    path_in_repo="metadata.jsonl",
    repo_id="tiptoghosh/food-recipes-15k",
    repo_type="dataset"
)